In [1]:
import tkinter as tk
from tkinter import messagebox
import cv2
import os
from PIL import Image
import numpy as np
import mysql.connector
from datetime import datetime
import time


In [2]:

# Initialize Tkinter window
window = tk.Tk()
window.title("Face Recognition System")
window.geometry("600x400")

# Frames for organizing the layout
input_frame = tk.Frame(window, padx=10, pady=10)
input_frame.grid(row=0, column=0, sticky="w")

button_frame = tk.Frame(window, padx=10, pady=10)
button_frame.grid(row=1, column=0, sticky="w")

# Labels and entry fields in input frame
tk.Label(input_frame, text="Name", font=("Arial", 12)).grid(column=0, row=0, sticky="w")
t1 = tk.Entry(input_frame, width=30, bd=2)
t1.grid(column=1, row=0, padx=5, pady=5)

tk.Label(input_frame, text="Age", font=("Arial", 12)).grid(column=0, row=1, sticky="w")
t2 = tk.Entry(input_frame, width=30, bd=2)
t2.grid(column=1, row=1, padx=5, pady=5)

tk.Label(input_frame, text="Address", font=("Arial", 12)).grid(column=0, row=2, sticky="w")
t3 = tk.Entry(input_frame, width=30, bd=2)
t3.grid(column=1, row=2, padx=5, pady=5)

# Button functions and layout in button frame
def train_classifier():
    data_dir = "D:/xampp/htdocs/PBL/facerecogeneration/data"
    path = [os.path.join(data_dir, f) for f in os.listdir(data_dir)]
    faces, ids = [], []

    for image in path:
        img = Image.open(image).convert('L')
        image_np = np.array(img, 'uint8')
        id = int(os.path.split(image)[1].split(".")[1])
        faces.append(image_np)
        ids.append(id)
    ids = np.array(ids)

    clf = cv2.face.LBPHFaceRecognizer_create()
    clf.train(faces, ids)
    clf.write("classifier.xml")
    messagebox.showinfo('Result', 'Training dataset completed!')

train_button = tk.Button(button_frame, text="Training", font=("Arial", 12), bg='orange', fg='red', command=train_classifier)
train_button.grid(column=0, row=0, padx=10, pady=10)

def detect_face():
    # Face detection function with tkinter message box
    faceCascade = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")
    clf = cv2.face.LBPHFaceRecognizer_create()
    clf.read("classifier.xml")

    video_capture = cv2.VideoCapture(0)
    while True:
        ret, img = video_capture.read()
        gray_image = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = faceCascade.detectMultiScale(gray_image, 1.3, 5)
        
        for (x, y, w, h) in faces:
            cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 2)
        cv2.imshow("Face Detection", img)
        if cv2.waitKey(1) == 13:
            break
    
    video_capture.release()
    cv2.destroyAllWindows()

detect_button = tk.Button(button_frame, text="Detect the face", font=("Arial", 12), bg='green', fg='white', command=detect_face)
detect_button.grid(column=1, row=0, padx=10, pady=10)

def generate_dataset():
    if not all([t1.get(), t2.get(), t3.get()]):
        messagebox.showinfo('Result', 'Please provide complete details of the user')
        return

    mydb = mysql.connector.connect(host="localhost", user="root", passwd="", database="authorized_user")
    mycursor = mydb.cursor()
    mycursor.execute("SELECT * FROM siswa")
    id = len(mycursor.fetchall()) + 1

    sql = "INSERT INTO siswa(id, Name, Age, Address) VALUES (%s, %s, %s, %s)"
    val = (id, t1.get(), t2.get(), t3.get())
    mycursor.execute(sql, val)
    mydb.commit()

    face_classifier = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")

    def face_cropped(img):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = face_classifier.detectMultiScale(gray, 1.3, 5)
        for (x, y, w, h) in faces:
            return img[y:y + h, x:x + w]
        return None

    cap = cv2.VideoCapture(0)
    img_id = 0
    while True:
        ret, frame = cap.read()
        cropped_face = face_cropped(frame)
        if cropped_face is not None:
            img_id += 1
            face = cv2.resize(cropped_face, (200, 200))
            face = cv2.cvtColor(face, cv2.COLOR_BGR2GRAY)
            file_name_path = f"data/user.{id}.{img_id}.jpg"
            cv2.imwrite(file_name_path, face)
            cv2.imshow("Cropped Face", face)
            if cv2.waitKey(1) == 13 or img_id == 200:
                break
    cap.release()
    cv2.destroyAllWindows()
    messagebox.showinfo('Result', 'Generating dataset completed!')

dataset_button = tk.Button(button_frame, text="Generate dataset", font=("Arial", 12), bg='pink', fg='black', command=generate_dataset)
dataset_button.grid(column=2, row=0, padx=10, pady=10)

window.mainloop()
